In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import os
import warnings
from scipy.stats import spearmanr
from scipy.stats import pearsonr
from scipy import stats
%matplotlib inline
plt.rcParams['pdf.fonttype'] = 42  
warnings.filterwarnings('ignore')

In [ ]:
data_path = os.path.dirname(os.getcwd()) + '/data'
figure_path = os.path.dirname(os.getcwd()) + '/figures'

In [ ]:
# Load differential expression results
datasets = {
    'bcp_all': 'Table S5 BCP-ALL vs Control',
    'aml': 'Table S4 AML vs Control', 
    't_all': 'Table S6 T-ALL vs Control',
    'leukemia': 'Table S3 Leukemia vs Control'
}

# Process differential expression results for each leukemia subtype
processed_data = {}
for name, sheet_name in datasets.items():
    df = pd.read_excel(data_path + '/results/results.xlsx', sheet_name=sheet_name, skiprows=1)
    
    # Filter 
    df = df[(df['adj.P.Val'] < 0.05) & (abs(df['Log2FC']) >= 1.5)]
    
    # Count up/down regulated proteins for summary
    upregulated = (df['Log2FC'] >= 1.5).sum()
    downregulated = (df['Log2FC'] <= -1.5).sum()
    
    print(f"{name}: {upregulated} upregulated, {downregulated} downregulated, {len(df)} total")
    processed_data[name] = df

# Extract individual dataframes
bcp_all = processed_data['bcp_all']
aml = processed_data['aml']
t_all = processed_data['t_all']
leukemia = processed_data['leukemia']

In [ ]:
# Find protein names that appear more than once
duplicate_proteins = bcp_all[bcp_all.duplicated(subset=['Protein'], keep=False)]
duplicate_proteins

In [ ]:
# Create clean lists of increased/decreased proteins for each immunophenotype
bcp_all_increased = bcp_all[bcp_all['Log2FC'] >= 1.5]['Protein'].tolist()
bcp_all_decreased = bcp_all[bcp_all['Log2FC'] <= -1.5]['Protein'].tolist()

aml_increased = aml[aml['Log2FC'] >= 1.5]['Protein'].tolist()
aml_decreased = aml[aml['Log2FC'] <= -1.5]['Protein'].tolist()

t_all_increased = t_all[t_all['Log2FC'] >= 1.5]['Protein'].tolist()
t_all_decreased = t_all[t_all['Log2FC'] <= -1.5]['Protein'].tolist()

# Print to verify
print(f"BCP-ALL: {len(bcp_all_increased)} increased, {len(bcp_all_decreased)} decreased")
print(f"AML: {len(aml_increased)} increased, {len(aml_decreased)} decreased")
print(f"T-ALL: {len(t_all_increased)} increased, {len(t_all_decreased)} decreased")

In [ ]:
print(len(set(bcp_all_increased)))
print(len(set(aml_increased)))
print(len(set(t_all_increased)))

### Clinical information

In [ ]:
clin_info = pd.read_excel(data_path + '/results/results.xlsx', sheet_name='Table S1 - Sample QC', skiprows=1)

colors = {
    'AML': "#EFB2D1",      # Yellow/gold
    'BCP-ALL': "#447597",  # Orange
    'T-ALL': '#87CCB2'     # Red
}

# Clinical variables to plot
clinical_vars = ['White blood cells', 'Blasts', 'Hemoglobin', 'Platelets']

# Remove Control patients from the dataframe
clin_filtered = clin_info[clin_info['Immunophenotype'] != 'Control'].copy()

# Rename B-ALL to BCP-ALL in the dataframe for consistency
clin_filtered['Immunophenotype'] = clin_filtered['Immunophenotype'].replace('B-ALL', 'BCP-ALL')

In [ ]:
# Set up the plotting style
plt.style.use('default')
sns.set_palette("husl")

# Define normal reference ranges for pediatric patients (6 months - 17 years)
# Combining ranges and using the most inclusive bounds
normal_ranges = {
    'White blood cells': (3.5, 14.0),  # 6 months-8 years: 3.5-14, 9-17 years: 4.1-12
    'Platelets': (150, 590),  # 6 months-10 years: 210-590, 11-17 years: 190-460 -> using 150-590 to be inclusive
    'Hemoglobin': (107, 170),  # pediatric range
    'Blasts': None  # No normal range - should be 0 in healthy individuals
}

# Create subplots - 2x2 grid for the 4 clinical variables
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
axes = axes.ravel()  # Flatten for easier indexing

# Plot density curves for each clinical variable
for i, var in enumerate(clinical_vars):
    ax = axes[i]
    
    # Add normal range shading if available
    if normal_ranges.get(var) is not None:
        min_val, max_val = normal_ranges[var]
        ax.axvspan(min_val, max_val, alpha=0.35, color='#A6A6A6', 
                  label='Normal range (pediatric)', zorder=0)
    
    # Get data for each immunophenotype - now simplified!
    for immuno_type in ['AML', 'BCP-ALL', 'T-ALL']:
        data = clin_filtered[clin_filtered['Immunophenotype'] == immuno_type][var].dropna()
        
        if len(data) > 0:
            # Special handling for percentage data (Blasts)
            if var == 'Blasts':
                sns.kdeplot(data=data, ax=ax, color=colors[immuno_type], 
                           label=immuno_type, linewidth=2.5, alpha=0.8,
                           clip=(0, 100), bw_adjust=0.8)  # Clip to 0-100% and adjust bandwidth
            else:
                # Regular clipping for other variables
                sns.kdeplot(data=data, ax=ax, color=colors[immuno_type], 
                           label=immuno_type, linewidth=2.5, alpha=0.8,
                           clip=(0, None))  # Clip to only show x >= 0
    
    # Set appropriate x-axis limits
    if var == 'Blasts':
        ax.set_xlim(0, 100)  # Force 0-100% for blasts
    else:
        ax.set_xlim(left=0)  # Start at 0 for other variables
    
    # Customize each subplot
    ax.set_ylabel('Density', fontsize=12)
    ax.tick_params(axis='both', which='major', labelsize=10)
    ax.grid(True, alpha=0.3, linestyle='--')
    
    # Add units to axis labels
    if var == 'White blood cells':
        ax.set_xlabel('White blood cells [×10⁹/L]', fontsize=12)
    elif var == 'Platelets':
        ax.set_xlabel('Platelets [×10⁹/L]', fontsize=12)
    elif var == 'Hemoglobin':
        ax.set_xlabel('Hemoglobin [g/L]', fontsize=12)
    elif var == 'Blasts':
        ax.set_xlabel('Blasts (%)', fontsize=12)
    
    # Remove individual legends
    if ax.get_legend():
        ax.get_legend().remove()
    
    # Remove top and right spines for cleaner look
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_linewidth(1.2)
    ax.spines['bottom'].set_linewidth(1.2)

# Adjust overall layout
plt.suptitle('Clinical Characteristics by Immunophenotype\nwith Pediatric Reference Ranges', 
             fontsize=14, y=1.05)

# Create custom legend
from matplotlib.patches import Patch
legend_elements = []

# Add immunophenotype colors
for immuno_type in ['AML', 'BCP-ALL', 'T-ALL']:
    legend_elements.append(plt.Line2D([0], [0], color=colors[immuno_type], 
                                    linewidth=2.5, label=immuno_type))

# Add reference range indicators
legend_elements.append(Patch(facecolor='#A6A6A6', alpha=0.35, 
                           label='"Normal" range (6m-17y)'))

# Add the legend
fig.legend(handles=legend_elements, bbox_to_anchor=(0.2, 0.85), 
           ncol=1, fontsize=10, frameon=True, fancybox=True, shadow=True)

# Ensure proper spacing between subplots
plt.tight_layout()

# Adjust layout to make room for the legend and title
plt.subplots_adjust(top=0.88, right=0.85)

# Save the figure
plt.savefig(figure_path + '/clinical_characteristics_with_references.png', 
           dpi=300, bbox_inches='tight')
plt.savefig(figure_path + '/clinical_characteristics_with_references.pdf', 
           dpi=300, bbox_inches='tight')
plt.show()

# Enhanced summary statistics with reference range context 
print("\nSummary Statistics by Immunophenotype with Reference Context:")
print("=" * 65)

for var in clinical_vars:
    print(f"\n{var}:")
    print("-" * len(var))
    
    # Print normal range if available
    if normal_ranges.get(var) is not None:
        min_val, max_val = normal_ranges[var]
        print(f"  Normal range (6m-17y): {min_val}-{max_val}")
    elif var == 'Blasts':
        print(f"  Normal range: 0% (no blasts in peripheral blood)")
    
    for immuno_type in ['AML', 'BCP-ALL', 'T-ALL']:
        data = clin_filtered[clin_filtered['Immunophenotype'] == immuno_type][var].dropna()
        
        if len(data) > 0:
            # Calculate percentage of values in normal range
            if normal_ranges.get(var) is not None:
                min_val, max_val = normal_ranges[var]
                in_range = ((data >= min_val) & (data <= max_val)).sum()
                pct_in_range = (in_range / len(data)) * 100
                
                print(f"  {immuno_type}: n={len(data)}, mean={data.mean():.1f}, "
                      f"median={data.median():.1f}, std={data.std():.1f}, "
                      f"in normal range: {in_range}/{len(data)} ({pct_in_range:.1f}%)")
            else:
                print(f"  {immuno_type}: n={len(data)}, mean={data.mean():.1f}, "
                      f"median={data.median():.1f}, std={data.std():.1f}")
        else:
            print(f"  {immuno_type}: No data available")

In [ ]:
# Update normal ranges - use log scale for WBC
normal_ranges = {
    'White blood cells': (np.log1p(3.5), np.log1p(14.0)),  # Log-transformed
    'Platelets': (150, 590),
    'Hemoglobin': (107, 170),
    'Blasts': None
}

# Create a transformed copy of WBC data
clin_filtered['WBC_log'] = np.log1p(clin_filtered['White blood cells'])

# Correct order: WBC, Blasts, Hemoglobin, Platelets
clinical_vars_plot = ['WBC_log', 'Blasts', 'Hemoglobin', 'Platelets']
var_labels = {
    'WBC_log': 'White blood cells (log-transformed) [×10⁹/L]',
    'Blasts': 'Blasts (%)',
    'Hemoglobin': 'Hemoglobin [g/L]',
    'Platelets': 'Platelets [×10⁹/L]'
}

# Create subplots
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
axes = axes.ravel()

# Dictionary to store axes for later retrieval
saved_axes = {}

for i, var in enumerate(clinical_vars_plot):
    ax = axes[i]
    
    range_key = 'White blood cells' if var == 'WBC_log' else var
    
    if normal_ranges.get(range_key) is not None:
        min_val, max_val = normal_ranges[range_key]
        ax.axvspan(min_val, max_val, alpha=0.35, color='#A6A6A6', 
                  label='Normal range (pediatric)', zorder=0)
    
    for immuno_type in ['AML', 'BCP-ALL', 'T-ALL']:
        data = clin_filtered[clin_filtered['Immunophenotype'] == immuno_type][var].dropna()
        
        if len(data) > 0:
            if var == 'Blasts':
                sns.kdeplot(data=data, ax=ax, color=colors[immuno_type], 
                           label=immuno_type, linewidth=2.5, alpha=0.8,
                           clip=(0, 100), bw_adjust=0.8)
            elif var == 'WBC_log':
                sns.kdeplot(data=data, ax=ax, color=colors[immuno_type], 
                           label=immuno_type, linewidth=2.5, alpha=0.8,
                           clip=(0.8, 7), bw_adjust=0.8)
            else:
                sns.kdeplot(data=data, ax=ax, color=colors[immuno_type], 
                           label=immuno_type, linewidth=2.5, alpha=0.8)
    
    # Set appropriate x-axis limits
    if var == 'Blasts':
        ax.set_xlim(0, 100)  # Force 0-100% for blasts
    elif var == 'WBC_log':
        ax.set_xlim(0.8, 7)  
    else:
        ax.set_xlim(left=0)  # Start at 0 for other variables
    
    ax.set_xlabel(var_labels[var], fontsize=12)
    ax.set_ylabel('Density', fontsize=12)
    ax.tick_params(axis='both', which='major', labelsize=10)
    ax.grid(True, alpha=0.3, linestyle='--')
    
    if ax.get_legend():
        ax.get_legend().remove()
    
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_linewidth(1.2)
    ax.spines['bottom'].set_linewidth(1.2)
    
    # Save WBC and Blasts axes
    if var == 'WBC_log':
        saved_axes['wbc'] = ax
    elif var == 'Blasts':
        saved_axes['blasts'] = ax

plt.suptitle('', fontsize=14, y=1.05)

legend_elements = [
    Patch(facecolor='#A6A6A6', alpha=0.35, label='REF (6m-17y)')
]

legend_elements += [
    plt.Line2D([0], [0],
               color=colors[immuno_type],
               linewidth=2.5,
               label=immuno_type)
    for immuno_type in ['AML', 'T-ALL', 'BCP-ALL']
]

fig.legend(handles=legend_elements, bbox_to_anchor=(0.21, 0.88), 
           ncol=1, fontsize=10, frameon=True, fancybox=True, shadow=True)

plt.tight_layout()
plt.subplots_adjust(top=0.88, right=0.85)

plt.savefig(figure_path + '/clinical_characteristics_with_references2.png', 
           dpi=300, bbox_inches='tight')
plt.savefig(figure_path + '/clinical_characteristics_with_references2.pdf', 
           dpi=300, bbox_inches='tight')
plt.show()

# Access later with: saved_axes['wbc'] and saved_axes['blasts']

### Analysing cilincal variables and biomarkers we discovered

In [ ]:
# Load and preprocess Olink proteomics data
olink = pd.read_csv(data_path + '/raw/olink.csv', delimiter=';')
olink = olink[['SampleID', 'Assay', 'NPX']]  # Keep only essential columns
print('Olink data - number of unique Sample IDs', len(set(olink['SampleID'])))

# Pivot data from long to wide format (samples as rows, proteins as columns)
pivoted_data = olink.pivot_table(index='SampleID', columns='Assay', values='NPX', aggfunc='mean')
pivoted_data.columns.name = None
pivoted_data = pivoted_data.reset_index()

# Load sample phenotype information
pheno = pd.read_excel(data_path + '/raw/MASTER_PHENO_FILE_AML_ALL_controls_20250607.xlsx')

# Clean immunopheno column - replace missing values with 'CTRL' (control)
pheno['immunopheno'] = pheno['immunopheno'].fillna('CTRL').replace(['nan', 'NaN', None], 'CTRL')
pheno['SampleID'] = pheno['sample_id']
print('Pheno data - number of unique Sample IDs', len(set(pheno['SampleID'])))
pheno = pheno[['SampleID', 'public_id', 'immunopheno', 'Subtype']]

# Merge proteomics data with phenotype information
merged = pd.merge(pivoted_data, pheno, on='SampleID', how='inner')

# Remove specific patients (quality control exclusions)
removed_patients = ['AML_101','AML_139', 'ALL_920', 'K-023']
final_df = merged[~merged['public_id'].isin(removed_patients)]
print('Merged data - number of unique Sample IDs', len(set(final_df['SampleID'])))

print('----------')
print('Sample distribution across immunophenotypes')
counts = final_df['immunopheno'].value_counts()
print(counts)

In [ ]:
# Drop SampleID and rename public_id to SampleID in one step
final_df = final_df.drop('SampleID', axis=1).rename(columns={'public_id': 'SampleID'})

clin_variables = clin_filtered.copy()
clin_variables['SampleID'] = clin_variables['Public Sample ID']
clin_variables = clin_variables[['SampleID', 'White blood cells', 'Blasts', 'Platelets', 'Hemoglobin']]

# Merge proteomics data with phenotype information
clin_merged = pd.merge(final_df, clin_variables, on='SampleID', how='inner')
clin_merged['immunopheno'] = clin_merged['immunopheno'].replace('B-ALL', 'BCP-ALL')

In [ ]:
clinical_cols = ['SampleID', 'immunopheno', 'Subtype', 'White blood cells', 
                     'Blasts', 'Platelets', 'Hemoglobin']
protein_cols = [col for col in clin_merged.columns if col not in clinical_cols]

def transform_clinical_variables(df):
    """Transform clinical variables for better correlation analysis."""
    df_transformed = df.copy()
    
    # Log transform White blood cells (highly skewed)
    wbc_mask = (~df_transformed['White blood cells'].isna()) & (df_transformed['White blood cells'] > 0)
    df_transformed.loc[wbc_mask, 'White blood cells'] = np.log1p(df_transformed.loc[wbc_mask, 'White blood cells'])
    
    # Log transform Platelets (right-skewed, 0-400 range)
    platelet_mask = (~df_transformed['Platelets'].isna()) & (df_transformed['Platelets'] > 0)
    df_transformed.loc[platelet_mask, 'Platelets'] = np.log1p(df_transformed.loc[platelet_mask, 'Platelets'])    
    return df_transformed

In [ ]:
def plot_npx_vs_clinical(df, protein_cols, figure_path=None, axes=None):
    import numpy as np
    import matplotlib.pyplot as plt
    from scipy.stats import pearsonr
    from matplotlib.patches import Patch

    # Fixed legend order to match the combination figure
    legend_order = ['AML', 'T-ALL', 'BCP-ALL']

    # Calculate median NPX per patient (avoid SettingWithCopy surprises)
    df = df.copy()
    df['median_npx'] = df[protein_cols].median(axis=1)

    clinical_vars = ['White blood cells', 'Blasts', 'Platelets', 'Hemoglobin']

    # Create figure only if axes not provided
    if axes is None:
        fig, axes = plt.subplots(1, 4, figsize=(18, 4))
        axes = axes.flatten()
        standalone = True
    else:
        standalone = False

    for i, clinical_var in enumerate(clinical_vars):
        ax = axes[i]

        # Collect legend labels with correlation stats 
        legend_labels = []

        # Plot for each immunophenotype in a fixed order
        for immuno in legend_order:
            subset = df[(df['immunopheno'] == immuno) &
                        (~df[clinical_var].isna()) &
                        (~df['median_npx'].isna())]

            if len(subset) > 0:
                ax.scatter(
                    subset[clinical_var], subset['median_npx'],
                    color=colors.get(immuno, "#474646"),
                    edgecolors='grey',
                    s=60,
                    alpha=0.9,
                    linewidth=0.5
                )

                # Correlation + optional regression line (matching later style)
                if len(subset) >= 3:
                    corr, p_val = pearsonr(subset[clinical_var], subset['median_npx'])

                    # regression line
                    z = np.polyfit(subset[clinical_var], subset['median_npx'], 1)
                    p = np.poly1d(z)
                    x_line = np.linspace(subset[clinical_var].min(),
                                         subset[clinical_var].max(), 100)
                    ax.plot(
                        x_line, p(x_line),
                        color=colors.get(immuno, "#474646"),
                        linestyle='--',
                        linewidth=2,
                        alpha=0.7
                    )

                    # significance stars
                    if p_val < 0.001:
                        stars = ' ***'
                    elif p_val < 0.01:
                        stars = ' **'
                    elif p_val < 0.05:
                        stars = ' *'
                    else:
                        stars = ' (ns)'

                    label = f'{immuno}: r={corr:.2f}{stars}'
                    legend_labels.append((immuno, label))
                else:
                    legend_labels.append((immuno, immuno))

        # Set x-axis labels with units
        if clinical_var == 'White blood cells':
            ax.set_xlabel('White blood cells (log-transformed) [×10⁹/L]')
        elif clinical_var == 'Blasts':
            ax.set_xlabel('Blasts (%)')
        elif clinical_var == 'Platelets':
            ax.set_xlabel('Platelets (log-transformed)')
        elif clinical_var == 'Hemoglobin':
            ax.set_xlabel('Hemoglobin [g/L]')

        ax.set_ylabel('Median NPX')
        ax.grid(True, alpha=0.3)

        # Build legend in the same way as earlier
        legend_dict = dict(legend_labels)
        legend_elements = []
        for immuno in legend_order:
            if immuno in legend_dict:
                legend_elements.append(
                    Patch(facecolor=colors[immuno], label=legend_dict[immuno])
                )

        # If you only want legend on the first subplot, use: if i == 0:
        ax.legend(handles=legend_elements, fontsize=9, loc='best', frameon=True)

    # Only handle figure-level stuff if standalone
    if standalone:
        plt.suptitle('', fontsize=14, fontweight='bold')
        plt.tight_layout()

        if figure_path:
            plt.savefig(f'{figure_path}/median_npx_vs_clinical.png', dpi=300, bbox_inches='tight')
            plt.savefig(f'{figure_path}/median_npx_vs_clinical.pdf', dpi=300, bbox_inches='tight')

        plt.show()

clin_transformed = transform_clinical_variables(clin_merged)
plot_npx_vs_clinical(clin_transformed, protein_cols, figure_path)

### Protein correlations individually with clinical characteristics

In [ ]:
def analyze_protein_correlations(df, clinical_var='White blood cells'):
    results = []
    
    for subtype in ['AML', 'BCP-ALL', 'T-ALL']:
        subtype_data = df[df['immunopheno'] == subtype]
        
        for protein in protein_cols:  # Uses global variable
            valid_data = subtype_data[[clinical_var, protein]].dropna()
            
            if len(valid_data) > 2:
                try:
                    corr, p_val = pearsonr(valid_data[clinical_var], valid_data[protein])
                    results.append({
                        'protein': protein,
                        'correlation': corr,
                        'p_value': p_val,
                        'immunopheno': subtype,
                        'n_samples': len(valid_data)
                    })
                except:
                    continue
    
    return pd.DataFrame(results)  

wbc_results = analyze_protein_correlations(clin_merged, 'White blood cells')
blast_results = analyze_protein_correlations(clin_merged, 'Blasts')

In [ ]:
def plot_de_correlations_simple(wbc_results, blast_results, 
                                bcp_increased, bcp_decreased,
                                aml_increased, aml_decreased,
                                tall_increased, tall_decreased,
                                figure_path=None):
    
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # Define what to plot for each subplot
    plot_specs = [
        # (data, immuno, protein_list, title, ax)
        (wbc_results, 'BCP-ALL', bcp_increased, 'WBC vs Increased (BCP-ALL)', axes[0, 0], 'BCP-ALL'),
        (wbc_results, 'AML', aml_increased, 'WBC vs Increased (AML)', axes[0, 0], 'AML'),
        (wbc_results, 'T-ALL', tall_increased, 'WBC vs Increased (T-ALL)', axes[0, 0], 'T-ALL'),
        
        (wbc_results, 'BCP-ALL', bcp_decreased, 'WBC vs Decreased (BCP-ALL)', axes[0, 1], 'BCP-ALL'),
        (wbc_results, 'AML', aml_decreased, 'WBC vs Decreased (AML)', axes[0, 1], 'AML'),
        (wbc_results, 'T-ALL', tall_decreased, 'WBC vs Decreased (T-ALL)', axes[0, 1], 'T-ALL'),
        
        (blast_results, 'BCP-ALL', bcp_increased, 'Blasts vs Increased (BCP-ALL)', axes[1, 0], 'BCP-ALL'),
        (blast_results, 'AML', aml_increased, 'Blasts vs Increased (AML)', axes[1, 0], 'AML'),
        (blast_results, 'T-ALL', tall_increased, 'Blasts vs Increased (T-ALL)', axes[1, 0], 'T-ALL'),
        
        (blast_results, 'BCP-ALL', bcp_decreased, 'Blasts vs Decreased (BCP-ALL)', axes[1, 1], 'BCP-ALL'),
        (blast_results, 'AML', aml_decreased, 'Blasts vs Decreased (AML)', axes[1, 1], 'AML'),
        (blast_results, 'T-ALL', tall_decreased, 'Blasts vs Decreased (T-ALL)', axes[1, 1], 'T-ALL'),
    ]
    
    # Group by subplot
    subplot_data = {}
    for corr_data, immuno, proteins, title, ax, label in plot_specs:
        ax_id = id(ax)
        if ax_id not in subplot_data:
            subplot_data[ax_id] = {'ax': ax, 'title': title.split('(')[0].strip(), 'data': []}
        
        # Get correlations for this immunophenotype and these proteins
        subset = corr_data[
            (corr_data['immunopheno'] == immuno) & 
            (corr_data['protein'].isin(proteins))
        ]['correlation']
        
        subplot_data[ax_id]['data'].append((immuno, subset))
    
    # Plot each subplot
    for ax_id, info in subplot_data.items():
        ax = info['ax']
        
        # Plot in reverse order for proper layering
        for immuno, correlations in reversed(info['data']):
            if len(correlations) > 0:
                ax.hist(correlations, alpha=0.7, bins=15,
                       label=f'{immuno} (n={len(correlations)})',
                       color=colors[immuno], edgecolor='black', linewidth=0.5)
        
        ax.set_xlabel('Correlation Coefficient', fontsize=11)
        ax.set_ylabel('Count', fontsize=11)
        ax.set_title(info['title'], fontsize=12, fontweight='bold')
        ax.legend(fontsize=9)
        ax.axvline(0, color='black', linestyle='--', alpha=0.5, linewidth=1.5)
        ax.grid(True, alpha=0.3)
    
    # Set proper titles
    axes[0, 0].set_title('White Blood Cells vs Increased Proteins', fontsize=12, fontweight='bold')
    axes[0, 1].set_title('White Blood Cells vs Decreased Proteins', fontsize=12, fontweight='bold')
    axes[1, 0].set_title('Blasts vs Increased Proteins', fontsize=12, fontweight='bold')
    axes[1, 1].set_title('Blasts vs Decreased Proteins', fontsize=12, fontweight='bold')
    
    plt.tight_layout()
    
    if figure_path:
        plt.savefig(f'{figure_path}/de_protein_correlations.png', dpi=300, bbox_inches='tight')
        plt.savefig(f'{figure_path}/de_protein_correlations.pdf', dpi=300, bbox_inches='tight')
    
    plt.show()

# Usage
plot_de_correlations_simple(
    wbc_results, blast_results,
    bcp_all_increased, bcp_all_decreased,
    aml_increased, aml_decreased,
    t_all_increased, t_all_decreased,
    figure_path=figure_path
)

### Combination figure

In [ ]:
def create_combination_figure_simple(clinical_var, figure_path=None):

    fig, axes = plt.subplots(1, 4, figsize=(16, 4))
    legend_order = ['AML', 'T-ALL', 'BCP-ALL']
    
    # Determine which clinical variable and correlation results to use
    if clinical_var == 'White blood cells':
        var_key = 'WBC_log'
        corr_data = wbc_results
        var_label = 'White blood cells (log-transformed) [×10⁹/L]'
        var_title = 'White Blood Cells'
        filename = 'white_blood_cells'
    else:  # Blasts
        var_key = 'Blasts'
        corr_data = blast_results
        var_label = 'Blasts (%)'
        var_title = 'Blasts'
        filename = 'blasts'
    
    # ============================================================================
    # SUBPLOT 1: Clinical characteristics density plot
    # ============================================================================
    ax = axes[0]
    
    # Add normal range if applicable
    if clinical_var == 'White blood cells':
        min_val, max_val = np.log1p(3.5), np.log1p(14.0)
        ax.axvspan(min_val, max_val, alpha=0.35, color='#A6A6A6', 
                  label='REF (6m-17y)', zorder=0)
        clip_range = (0.8, 7)
        xlim = (0.8, 7)
    else:  # Blasts
        clip_range = (0, 100)
        xlim = (0, 100)
    
    # Plot density for each immunophenotype
    for immuno_type in ['BCP-ALL', 'T-ALL', 'AML']: 
        data = clin_filtered[clin_filtered['Immunophenotype'] == immuno_type][var_key].dropna()
        
        if len(data) > 0:
            if clinical_var == 'Blasts':
                sns.kdeplot(data=data, ax=ax, color=colors[immuno_type], 
                           label=immuno_type, linewidth=2.5, alpha=0.8,
                           clip=clip_range, bw_adjust=0.8)
            else:
                sns.kdeplot(data=data, ax=ax, color=colors[immuno_type], 
                           label=immuno_type, linewidth=2.5, alpha=0.8,
                           clip=clip_range, bw_adjust=0.8)
    
    ax.set_xlim(xlim)
    ax.set_xlabel(var_label, fontsize=11)
    ax.set_ylabel('Density', fontsize=11)
    ax.set_title(f'{var_title} Distribution', fontsize=12)
    ax.grid(True, alpha=0.3, linestyle='--')

    from matplotlib.lines import Line2D
    from matplotlib.patches import Patch

    legend_elements = []

    # Add REF first (only for WBC)
    if clinical_var == 'White blood cells':
        legend_elements.append(
            Patch(facecolor='#A6A6A6', alpha=0.35, label='REF (6m-17y)')
        )

    # Then add AML, T-ALL, BCP-ALL in fixed order
    for immuno in legend_order:
        legend_elements.append(
            Line2D([0], [0],
                color=colors[immuno],
                linewidth=2.5,
                label=immuno)
        )

    ax.legend(handles=legend_elements, fontsize=9, loc='best')

    # ============================================================================
    # SUBPLOT 2: Median NPX vs clinical variable
    # ============================================================================
    ax = axes[1]
    
    # Collect legend labels with correlation stats
    legend_labels = []
    
    # Plot for each immunophenotype
    for immuno_type in ['BCP-ALL', 'T-ALL', 'AML']:
        subset = clin_transformed[clin_transformed['immunopheno'] == immuno_type]
        valid_data = subset[[clinical_var, 'median_npx']].dropna()
        
        if len(valid_data) > 0:
            ax.scatter(valid_data[clinical_var], valid_data['median_npx'], 
                      color=colors[immuno_type], 
                      s=60, edgecolors='grey', alpha= 0.9, linewidth=0.5)
            
            # Calculate correlation and add regression line
            if len(valid_data) >= 3:
                corr, p_val = pearsonr(valid_data[clinical_var], valid_data['median_npx'])
                
                # Add regression line
                z = np.polyfit(valid_data[clinical_var], valid_data['median_npx'], 1)
                p = np.poly1d(z)
                x_line = np.linspace(valid_data[clinical_var].min(), 
                                    valid_data[clinical_var].max(), 100)
                ax.plot(x_line, p(x_line), color=colors[immuno_type], 
                       linestyle='--', linewidth=2, alpha=0.7)
                
                # Determine significance stars
                if p_val < 0.001:
                    stars = ' ***'
                elif p_val < 0.01:
                    stars = ' **'
                elif p_val < 0.05:
                    stars = ' *'
                else:
                    stars = ' (ns)'
                
                # Create label with correlation and significance
                label = f'{immuno_type}: r={corr:.2f}{stars}'
                legend_labels.append((immuno_type, label))
            else:
                legend_labels.append((immuno_type, immuno_type))
    
    ax.set_xlabel(var_label, fontsize=11)
    ax.set_ylabel('Median NPX', fontsize=11)
    ax.set_title(f'Median NPX vs {var_title}', fontsize=12)
    
    # Create custom legend with colored patches and correlation stats
    from matplotlib.patches import Patch
    legend_dict = dict(legend_labels)

    legend_elements = []
    for immuno in legend_order:
        if immuno in legend_dict:
            legend_elements.append(
                Patch(facecolor=colors[immuno],
                    label=legend_dict[immuno])
            )
    ax.legend(handles=legend_elements, fontsize=9, loc='best', frameon=True)
    ax.grid(True, alpha=0.3)
  
    # ============================================================================
    # SUBPLOT 3: Correlations with INCREASED proteins
    # ============================================================================
    ax = axes[2]
    
    for immuno in ['T-ALL', 'AML', 'BCP-ALL']:  # Reverse order for layering
        if immuno == 'BCP-ALL':
            proteins = bcp_all_increased
        elif immuno == 'AML':
            proteins = aml_increased
        else:  # T-ALL
            proteins = t_all_increased
        
        subset = corr_data[
            (corr_data['immunopheno'] == immuno) & 
            (corr_data['protein'].isin(proteins))
        ]['correlation']
        
        if len(subset) > 0:
            ax.hist(subset, alpha=0.7, bins=15,
                   label=f'{immuno}',
                   color=colors[immuno], edgecolor='black', linewidth=0.5)
    
    ax.set_xlabel('Correlation Coefficient', fontsize=11)
    ax.set_ylabel('Count', fontsize=11)
    ax.set_title(f'{var_title} vs Increased Proteins', fontsize=12)
    #ax.legend(fontsize=9, loc='best')
    ax.set_xlim(-1,1) 
    ax.axvline(0, color='black', linestyle='--', alpha=0.5, linewidth=1.5)
    ax.grid(True, alpha=0.3)
    
    # ============================================================================
    # SUBPLOT 4: Correlations with DECREASED proteins
    # ============================================================================
    ax = axes[3]
    
    for immuno in ['BCP-ALL','T-ALL','AML']: # ['T-ALL', 'AML', 'BCP-ALL']:  # Reverse order for layering
        if immuno == 'BCP-ALL':
            proteins = bcp_all_decreased
        elif immuno == 'AML':
            proteins = aml_decreased
        else:  # T-ALL
            proteins = t_all_decreased
        
        subset = corr_data[
            (corr_data['immunopheno'] == immuno) & 
            (corr_data['protein'].isin(proteins))
        ]['correlation']
        
        if len(subset) > 0:
            ax.hist(subset, alpha=0.7, bins=15,
                   label=f'{immuno}',
                   color=colors[immuno], edgecolor='black', linewidth=0.5)
    
    ax.set_xlabel('Correlation Coefficient', fontsize=11)
    ax.set_ylabel('Count', fontsize=11)
    ax.set_title(f'{var_title} vs Decreased Proteins', fontsize=12)
    ax.set_xlim(-1, 1) 
    ax.axvline(0, color='black', linestyle='--', alpha=0.5, linewidth=1.5)
    ax.grid(True, alpha=0.3)

    from matplotlib.patches import Patch

    legend_elements = [
        Patch(facecolor=colors[immuno], label=immuno)
        for immuno in legend_order
    ]

    ax.legend(handles=legend_elements, fontsize=9, loc='best')
    
    # Add panel labels
    panel_labels = ['A', 'B', 'C', 'D']
    for i, (ax_item, label) in enumerate(zip(axes, panel_labels)):
        ax_item.text(-0.15, 1.15, label, transform=ax_item.transAxes, 
                    fontsize=16, fontweight='bold', va='top')
    
    # Overall title
    plt.suptitle('',#f'{var_title}', 
                fontsize=16, fontweight='bold', y=1.02)
    
    plt.tight_layout()
    
    # Save figure
    if figure_path:
        plt.savefig(f'{figure_path}/combination_{filename}.png', dpi=300, bbox_inches='tight')
        plt.savefig(f'{figure_path}/combination_{filename}.pdf', dpi=300, bbox_inches='tight')
        plt.savefig(f'{figure_path}/combination_{filename}.svg', dpi=300, bbox_inches='tight')
    
    plt.show()

In [ ]:
create_combination_figure_simple('White blood cells', figure_path=figure_path)
create_combination_figure_simple('Blasts', figure_path=figure_path)